# Introducción a los LLMs Locales con Ollama
 
¡Es hora de ponerse prácticos!
 
Este notebook demuestra cómo ejecutar un Modelo de Lenguaje Grande (LLM) de forma local usando Ollama y LangChain. Todo correrá directamente en tu computador.
 
Ollama nos permite ejecutar LLMs localmente en nuestra máquina. LangChain es un SDK (biblioteca de código reutilizable) que facilita la interacción con LLMs.

![Alt text](figs/01/ollama_langchain_architecture.svg) 
## 1. Prerrequisitos
 
Antes de ejecutar este notebook, necesitas instalar Ollama y descargar un modelo. Sigue los pasos a continuación.
 
### 1.1 Instalación de Ollama
 
**macOS:**
- Descarga el instalador desde [ollama.ai](https://ollama.ai)
- Abre el archivo descargado y sigue las instrucciones de instalación
 
**Windows:**
- Descarga el instalador desde [ollama.ai](https://ollama.ai)
- Ejecuta el instalador y sigue las instrucciones
 
**Linux:**

In [ ]:
 %%bash
curl -fsSL https://ollama.ai/install.sh | sh

Verifica tu instalación (Windows, Mac y Linux):

In [ ]:
%%bash
ollama --version

### 1.2 Descarga del Modelo TinyLlama
 
Descarga el modelo TinyLlama (aproximadamente 600MB). Existe un amplio catálogo de modelos disponibles en Ollama, incluyendo DeepSeek y otros, pero TinyLlama es el más pequeño, lo cual es ideal para propósitos de demostración:


In [ ]:
%%bash
ollama pull tinyllama:1.1b

## 2. Configuración del Entorno Python
 
Instala los paquetes necesarios. Instalaremos LangChain, una de las bibliotecas más populares para interactuar con LLMs. langchain-ollama nos permite interactuar con modelos localmente.

In [ ]:
%%bash
pip install langchain langchain-ollama

## 3. Usando el LLM con LangChain
 
El código a continuación nos permite interactuar con el modelo tinyllama alojado localmente en nuestros computadores. Simplemente enviamos el siguiente mensaje al modelo: "I love programming".
 
Lee los comentarios del código si quieres entender exactamente qué está pasando.

In [ ]:
# Importamos las bibliotecas necesarias
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage
 
# Inicializamos el LLM. Aquí elegimos el modelo y la temperatura
# (que controla la aleatoriedad de la salida). llm es lo que usaremos para llamar al modelo.
llm = ChatOllama(
    model="tinyllama:1.1b",
    temperature=0,  # 0 para salidas más deterministas
)
 
# Debemos proveer un arreglo de mensajes al modelo. El primer mensaje es siempre el mensaje
# de sistema, que le indica al modelo cómo comportarse. El segundo es el mensaje del usuario,
# que es lo que queremos preguntarle al modelo.
messages = [
    (
        "system",
        "You are a helpful assistant",
    ),
    ("human", "I love programming."),
]
 
# Enviamos el arreglo de mensajes al modelo y obtenemos la respuesta.
# La respuesta es un objeto AIMessage que contiene el texto generado, que imprimiremos a continuación.
ai_msg = llm.invoke(messages)
 
# Mostramos la respuesta
print(ai_msg.content)

# 4. Probando Diferentes Prompts
 
Probemos un par de ejemplos para ver qué puede hacer el modelo, cambiando el arreglo de mensajes:

In [ ]:
# Pedimos una explicación
messages = [
    (
        "system",
        "You are a helpful and informative AI assistant.",
    ),
    ("human", "Explain the concept of a neural network in simple terms."),
]
 
ai_msg = llm.invoke(messages)
print(ai_msg.content)

In [ ]:
# Pedimos algo de código
messages = [
    (
        "system",
        "You are an expert Python programmer.",
    ),
    ("human", "Write a function to check if a string is a palindrome."),
]
 
ai_msg = llm.invoke(messages)
print(ai_msg.content)

## 5. Creando un Chatbot Básico
 
Vamos a crear una interfaz de chat simple para interactuar con el modelo. Cada celda representará una interacción. Sigue los pasos a continuación; tendrán más sentido a medida que los vayas ejecutando.

In [9]:
# Configuración: ejecuta esta celda primero
from langchain_ollama import ChatOllama

In [ ]:
# Inicializamos el LLM
llm = ChatOllama(
    model="tinyllama:1.1b",
    temperature=0
)
 
# Prompt de sistema: define el comportamiento del asistente
system_prompt = "You are a helpful assistant"
 
print("-" * 30)

Creemos una celda para nuestra primera interacción, diciéndole al modelo cuál es nuestro color favorito. Ten en cuenta que la respuesta puede ser extraña, ya que no estamos usando el mejor modelo disponible. Pero debería indicar que entendió que tu color favorito es el azul."""

In [ ]:
# Primera interacción
user_message = "My favorite color is blue."
 
# Creamos mensajes nuevos solo para esta interacción (sin historial previo)
messages = [
    ("system", system_prompt),
    ("human", user_message)
]
 
# Obtenemos la respuesta
ai_msg = llm.invoke(messages)
 
print(f"Tú: {user_message}")
print(f"Asistente: {ai_msg.content}")

Ahora hagamos una pregunta de seguimiento:

In [ ]:
# Segunda interacción
user_message = "What's my favorite color?"
 
# Creamos mensajes nuevos solo para esta interacción (sin historial previo)
messages = [
    ("system", system_prompt),
    ("human", user_message)
]
 
# Obtenemos la respuesta
ai_msg = llm.invoke(messages)
 
print(f"Tú: {user_message}")
print(f"Asistente: {ai_msg.content}")

Como puedes ver, ¡el modelo no recuerda nuestro color favorito! Podrías preguntarte por qué. Seguramente herramientas como ChatGPT pueden recordar el contexto completo de la conversación. Este es el concepto de memoria. El LLM, por sí solo, no recuerda conversaciones pasadas. Necesitamos diseñar nuestra aplicación para que lo haga.
 
Exploremos cómo hacerlo a continuación, y cómo herramientas como ChatGPT lo gestionan.

## 6. Entendiendo Cómo Funciona la Memoria de Conversación
 
La forma más sencilla de gestionar esto es pasar todo el historial de conversación al modelo, en lugar de solo el mensaje individual:

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
 
# Inicializamos el LLM como antes
llm = ChatOllama(
    model="tinyllama:1.1b",
    temperature=0
)
 
# Definimos el historial de conversación (siempre empieza con un prompt de sistema).
# conversation es una lista de mensajes. SystemMessage representa un mensaje de sistema
# y acepta el contenido como parámetro.
conversation = [SystemMessage(content="You are a helpful assistant")]
 
# Definimos una función llamada chat que toma la entrada del usuario como parámetro.
# Esta función se usará para interactuar con el modelo.
def chat(user_input: str):
    # 1) Agregamos el mensaje del usuario al historial.
    # Ahora la lista contiene el mensaje de sistema y el mensaje del usuario.
    conversation.append(HumanMessage(content=user_input))
 
    # 2) Llamamos al modelo con la lista completa de conversación
    ai_msg = llm.invoke(conversation)
 
    # 3) Mostramos el intercambio
    print(f"Tú: {user_input}")
    print(f"Asistente: {ai_msg.content}")
    print(f"[Longitud de conversación: {len(conversation) + 1} mensajes]")  # +1 por la respuesta del modelo
 
    # 4) Agregamos la respuesta del modelo al historial. Ahora la lista contiene el mensaje
    # de sistema, el mensaje del usuario y la respuesta del modelo, que se usará en la próxima interacción.
    conversation.append(ai_msg)
 
    return ai_msg.content
 
print("Chat Con Memoria")
print("-" * 30)

In [ ]:
"""Démosle nuevamente nuestro color favorito:"""
 
# Primera interacción: compartimos el color favorito
chat("My favorite color is blue")

In [ ]:
"""Ahora veamos si lo recuerda:"""
 
# Segunda interacción: ponemos a prueba la memoria
chat("What's my favorite color?")

¡Ahora el modelo debería recordar correctamente tu color favorito! Esto se debe a que estamos llevando el registro del historial de conversación.

## 7. El Desafío de la Ventana de Contexto
 
Aunque nuestra solución de memoria simple funciona inicialmente (porque pasamos todo el historial como entrada), tiene una limitación crítica: **la ventana de contexto**.
 
### Entendiendo la Ventana de Contexto
 
Cada modelo de lenguaje tiene una "ventana de contexto" fija: el número máximo de tokens (aproximadamente palabras o fragmentos de palabras) que puede procesar a la vez:
 
- **TinyLlama**: ~2,048 tokens (aproximadamente 1,500 palabras)
- **GPT-3.5**: ~4,096 tokens
- **GPT-4**: entre 8,192 y 128,000 tokens según la versión
- **Claude 3**: hasta 200,000 tokens
 
A medida que la conversación crece, eventualmente alcanzamos este límite. Cuando eso ocurre:
 
1. El modelo no puede ver mensajes más allá del tamaño de la ventana
2. Efectivamente "olvida" las partes más antiguas de la conversación
3. El procesamiento se vuelve más lento con contextos más grandes
4. Puedes recibir errores si superas la ventana de contexto
 
## 8. Soluciones al Problema de la Ventana de Contexto
 
En sistemas de producción como ChatGPT, varias técnicas abordan esta limitación:
 
### 1. Ventana de Contexto Deslizante
Este enfoque conserva solo los N mensajes más recientes, más el prompt de sistema.
 
**Ejemplo:** Si limitas a 10 mensajes y la conversación llega a 15, descartarías los 5 mensajes más antiguos (excepto el prompt de sistema).
 
Es como tener una conversación donde solo recuerdas lo dicho en los últimos minutos. Es simple, pero pierde toda la información anterior.
 
### 2. Resumen de Conversación (Summarization)
Esta técnica condensa las partes más antiguas de la conversación en resúmenes para ahorrar tokens.
 
**Ejemplo:** Después de 10 intercambios, el sistema podría reemplazar esos 20 mensajes con un único resumen: "El usuario preguntó sobre redes neuronales. El asistente las explicó y proporcionó un ejemplo en Python para detectar palíndromos."
 
Esto preserva los puntos clave mientras reduce significativamente el uso de tokens.
 
### 3. Gestión de Sesiones con Base de Datos
 
Este enfoque usa una base de datos para almacenar historiales completos de conversación asociados a sesiones únicas de usuario. Así funciona:
 
1. **Creación de sesión**: Cuando un usuario comienza a chatear, el sistema crea un ID de sesión único (como "session_abc123")
 
2. **Almacenamiento de mensajes**: Cada mensaje del usuario y del modelo se guarda en una tabla de la base de datos vinculada a ese ID de sesión
 
3. **Gestión de la ventana de contexto**: Al preparar el prompt para el LLM, el sistema:
   - Recupera todos los mensajes de la sesión
   - Aplica estrategias para ajustarse a la ventana de contexto (ventana deslizante o resumen)
   - Envía la conversación optimizada al modelo
   - Almacena la nueva respuesta de vuelta en la base de datos
 
**Ejemplo práctico:**
- Un usuario ha estado chateando durante horas con ChatGPT
- Su sesión "user_789" ahora tiene 200 mensajes en la base de datos
- Cuando envía el mensaje #201, el sistema:
  - Recupera los 200 mensajes anteriores de la base de datos
  - Selecciona los más relevantes para ajustarse a la ventana de contexto
  - Obtiene una respuesta del modelo
  - Agrega el mensaje #201 y la respuesta a la base de datos
  - Aunque el modelo solo "ve" la porción reciente, el historial completo permanece en la base de datos
 
Así es como servicios como ChatGPT pueden mantener "memoria" en conversaciones muy largas, incluso cuando cierras el navegador y vuelves más tarde.
 
## ¿Qué usa ChatGPT típicamente?
 
ChatGPT usa un enfoque híbrido que combina varias de estas técnicas:
 
1. **Técnica principal:** Usa una ventana de contexto muy grande (hasta 128K tokens en GPT-4o) y gestión de sesiones con base de datos, lo que le permite "recordar" conversaciones notablemente largas.
 
2. **Para conversaciones largas:** Cuando las conversaciones superan estos límites, emplea técnicas de ventana deslizante que priorizan:
   - El prompt de sistema e instrucciones
   - Los mensajes más recientes
   - Mensajes con alta densidad de información
   - Mensajes referenciados explícitamente por el usuario
 
3. **Compresión dinámica:** En algunas versiones, ChatGPT emplea algoritmos de compresión dinámica que resumen o eliminan selectivamente partes de la conversación menos relevantes para el intercambio actual.
 
Aunque estas soluciones ayudan, sigue existiendo un límite absoluto de lo que cualquier modelo puede "recordar" en una sola conversación. Por eso, incluso ChatGPT a veces "olvida" cosas mencionadas mucho antes en conversaciones muy largas.
 
## 9. Ventajas de los LLMs Locales
 
Usar LLMs locales con herramientas como Ollama tiene varias ventajas:
 
1. **Privacidad**: Tus datos nunca salen de tu computador
2. **Sin costos de API**: Ejecuta el modelo tanto como quieras sin pagar por consulta
3. **Uso sin conexión**: No necesitas internet una vez que el modelo está descargado
4. **Sin límites de uso**: Ejecuta tantas consultas como tu hardware pueda manejar
 
## 10. Recursos para Seguir Aprendiendo
 
- [Documentación de Ollama](https://github.com/ollama/ollama/blob/main/README.md)
- [Documentación de LangChain](https://python.langchain.com/docs/get_started/introduction)
- [Información sobre TinyLlama](https://github.com/jzhang38/TinyLlama)
"""